# Redes Neurais: Classificação com _Multilayer Perceptron_
Caio Ávila Paulo

In [1]:
# Importações Necessárias
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  MinMaxScaler
# Semente Aleatória
from numpy.random import seed
SEMENTE_ALEATORIA = 73
seed(SEMENTE_ALEATORIA)
random.seed(SEMENTE_ALEATORIA)
torch.manual_seed(SEMENTE_ALEATORIA);

# Introdução

No aprendizado de máquina supervisionado, existem dois principais tipos de problema: os de regressão, em que o alvo é um dado numérico, e os de classificação, em que ele é categórico. O funcionamento de uma rede neural _multilayer perceptron_ (MLP) no primeiro caso envolve a aplicação sequencial de funções de ativação sobre os dados de entrada, retornando dados de saída numéricos e atualizando-se os parâmetros de cada neurônio de acordo com sua influência em uma função de perda, a qual mede o quão próxima está a previsão dos dados de seu valor real.

No caso de dados categóricos, parece impossível relacionar esse retorno numérico com uma previsão de rótulo, quem dirá avaliar a precisão dela! No entanto, com algumas técnicas, é possível construir redes MLP para problemas de classificação, preservando em grande parte o raciocínio usado nas MLP de regressão. Neste _notebook_, serão criadas MLPs para classificação binária e multiclasse, usando Python puro e a biblioteca `PyTorch`, com ênfase nas semelhanças e diferenças em relação à MLP de regressão.

## Observação:

Será usado um _dataset_ didático com valores fictícios sobre personagens de RPG [1], e o código será construido principalmente com base no material do Prof. Daniel R. Cassar sobre MLPs de regressão, tanto em Python puro [2] quanto com o `torch`[3]. Como o objetivo deste _notebook_ é a didática sobre classificação com MLPs, não serão incluídas neles discussões mais gerais de aprendizado de máquina (otimização de hiperparâmetros, regularização, validação cruzada, tratamento e vazamento de dados etc.) No entanto, fique o leitor avisado de que todas essas técnicas podem ser usadas em MLPs classificadoras - e, especialmente, que os recursos aqui aprendidos e utilizados em dados "bobos" são perfeitamente utilizáveis em dados científicos!

In [2]:
df = pd.read_csv('RPG.csv')
display(df)

,Armor,Weapon,Physical,Magic,Level,FBoss,Class
0,45,83,45,89,77,True,Magician
1,48,88,82,65,71,True,Warrior
2,65,22,71,4,37,False,Tank
3,68,3,7,91,39,False,Sorcerer
4,68,9,49,21,27,False,Tank
...,...,...,...,...,...,...,...
995,27,43,73,90,67,True,Battlemage
996,49,69,81,30,50,True,Knight
997,72,63,19,72,61,True,Magician
998,55,37,75,27,66,False,Battlemage


# Problemas de Classificação

Tem-se um problema de classificação quando, a partir de determinados atributos, quer-se prever um alvo categórico. Ou seja, para cada dado inserido, a rede neural deve prever um rótulo ao qual o _target_ desse dado pertence. Dessa forma, algumas noções que se tem em problemas de regressão, nos quais se prevê um dado numérico, perdem o sentido.

Por exemplo, se nossa rede precisasse prever o 'Level' de um personagem e dissesse que ele é 2, mas soubéssemos que o valor verdadeiro é 70, essa seria uma previsão ruim, pois estaria muito distante da realidade. Com essas relações numéricas, é possível construir funções que avaliem a capacidade de previsão - o erro quadrático médio (MSE), por exemplo -, então treiná-las de forma a minimizar esse erro e avaliá-las com essa métrica.

Em uma MLP classificadora, será feito algo semelhante: também teremos uma função de perda a ser minimizada com a atualização dos parâmetros durante o treinamento, e ao final ainda poderemos aferir o desempenho comparando os dados reais de teste e suas previsões com uma métrica, mas essas funções serão aplicadas de uma forma menos "direta" que na regressão. Uma boa noção inicial é que, internamente, nossa rede continuará gerando números, porém esses valores não representarão o alvo em si, mas a probabilidade de se atribuir a ele cada um dos rótulos possíveis! Com base nessas probabilidades, podemos treinar a rede e prepará-la para realizar a classificação - nesse caso, ela não retornará as probabilidades, mas o rótulo que ela avaliou que é mais provável, pois o dado de saída de uma rede classificadora deve ser categórico.

A seguir, você verá como fazemos isso, em problemas de classificação tanto binária quanto multiclasse.

# Classificação Binária

Aquela na qual o alvo tem apenas dois rótulos possíveis. No mundo real, para dizer se um material é condutor ou isolante, ou se um indivíduo está ou não contaminado, podem ser usada classificação binária. Por exemplo, no _dataset_ didático a seguir, cada personagem de RPG possui vários atributos de poder ('Armor', 'Weapon', 'Physical', 'Magic', 'Level') distribuídos de 1 a 100 pontos, mas a coluna 'FBoss' aceita apenas True ou False: ou o personagem derrotou o chefão, ou ele não derrotou. Prever 'FBoss' com base nas outras colunas é um problema de classificação binária.

In [3]:
ATRIBUTOS = ['Armor', 'Weapon', 'Physical', 'Magic', 'Level']
TARGET = ['FBoss']
NUM_DADOS_DE_ENTRADA = len(ATRIBUTOS)
NUM_DADOS_DE_SAIDA = len(TARGET)
display(df.reindex(ATRIBUTOS+TARGET, axis=1))

,Armor,Weapon,Physical,Magic,Level,FBoss
0,45,83,45,89,77,True
1,48,88,82,65,71,True
2,65,22,71,4,37,False
3,68,3,7,91,39,False
4,68,9,49,21,27,False
...,...,...,...,...,...,...
995,27,43,73,90,67,True
996,49,69,81,30,50,True
997,72,63,19,72,61,True
998,55,37,75,27,66,False


Nesses casos, é comum trocar os rótulos por 1 e 0. Isso não transforma o problema em uma regressão, pois estamos interessados em medir apenas 1 **ou** 0, não qualquer número entre eles. Isso também não transforma o alvo em um dado categórico ordinal; 1 > 0 não implica True > False! Essa transformação serve apenas para melhor relacionar o rótulo com uma previsão de probabilidade. (Além disso, é sempre útil codificar os dados como numéricos porque certos métodos de bibliotecas como `torch` não aceitam _inputs_ de outro formato.)

Imagine que você entregou um dado de teste a sua rede neural. Se ela devolver uma probabilidade 0.97 de ser True, automaticamente ela dirá que a probabilidade de ser False é 0.03; isso quer dizer que sua rede classificadora está, com muita confiança, te devolvendo o rótulo "True". Se ela acertar, o palpite dela foi muito bom, mas se ela errar, o palpite foi muito ruim - ela chutou o caso contrário com quase 100% de certeza!

Quantificar esses palpites é muito simples quando se usa a troca de rótulos sugerida: o número fornecido por sua rede neural torna-se simplesmente a probabilidade que ela atribui ao caso positivo; então, quanto menor o módulo da diferença entre o valor real do rótulo e essa probabilidade, melhor é a previsão.

No Python, é muito simples transformar os valores de FBoss: com `.astype(int)`, True é automaticamente substituído por 1 e False por 0. Note que a substituição não _precisa_ ser essa; você poderia considerar False como o caso positivo (de rótulo 1), se quisesse.

In [4]:
df['FBoss'] = df['FBoss'].astype(int)
display(df.reindex(ATRIBUTOS+TARGET, axis=1))

,Armor,Weapon,Physical,Magic,Level,FBoss
0,45,83,45,89,77,1
1,48,88,82,65,71,1
2,65,22,71,4,37,0
3,68,3,7,91,39,0
4,68,9,49,21,27,0
...,...,...,...,...,...,...
995,27,43,73,90,67,1
996,49,69,81,30,50,1
997,72,63,19,72,61,1
998,55,37,75,27,66,0


## Divisão e Tratamento dos Dados
10% dos dados serão reservados ao conjunto de teste.

In [5]:
indices = df.index
TAMANHO_TESTE = 0.1

indices_treino, indices_teste = train_test_split(indices, test_size=TAMANHO_TESTE, random_state=SEMENTE_ALEATORIA)

df_treino = df.loc[indices_treino]
df_teste = df.loc[indices_teste]

X_treino = df_treino.reindex(ATRIBUTOS, axis=1)
y_treino = df_treino.reindex(TARGET, axis=1)

X_teste = df_teste.reindex(ATRIBUTOS, axis=1)
y_teste = df_teste.reindex(TARGET, axis=1)

Não é **necessário** normalizar os dados (especialmente no nosso caso, em que todos os atributos já estão na mesma escala), mas é uma boa prática. Como nesse notebook será usada a função sigmoide diversas vezes, optou-se por deixar os dados de entrada contidos em [0,1], de modo que a transformação entre os dados de entrada e os que são enviados para a primeira camada oculta não seja tão diferente da entre os que a primeira envia para a segunda (valores de entrada muito grandes fariam a sigmoide tender a 1 e poderiam atrasar a convergência em algumas iterações). A normalização por mínimos e máximos deixa os números exatamente nesse intervalo:

In [6]:
normalizador_atributos = MinMaxScaler()
normalizador_atributos.fit(X_treino)

X_treino_norm = normalizador_atributos.transform(X_treino)
X_teste_norm = normalizador_atributos.transform(X_teste)

Note que não faz sentido normalizar o alvo, já que ele é categórico. Além disso, seus valores já estão em [0,1].

Apenas para serem compatíveis com os passos seguintes desse notebook, que usam Python puro, transformaremos os dados de treino e teste em listas.

In [7]:
X_treino_norm_ = X_treino_norm.tolist()
y_treino_ = y_treino.to_numpy().ravel().tolist()

X_teste_norm_ = X_teste_norm.tolist()
y_teste_ = y_teste.to_numpy().ravel().tolist()

## Construção da Rede Neural em Python Puro
Para criar a MLP, são necessárias várias etapas - e vamos realizar cada uma delas em Python puro, para garantir que seja compreendido cada passo do funcionamento de uma rede de classificação binária! O código a seguir foi copiado do material didático do Prof. Daniel R. Cassar, modificando-se o mínimo possível para transformar uma rede regressora em classificadora.
### A classe `Valor`
Para usar _autograd_ e _backpropagation_, é necessário criar um objeto que guarde a "história" das operações que o criaram, possibilitando a construção do grafo computacional com as derivadas parciais. A única diferença em relação ao código original foi a inclusão do logaritmo nas operações, o qual será necessário durante o treinamento. Repare que foi usado um "plano de contingência" para log(0), pois é possível, ainda que improvável, que ele apareça, e não queremos que o código seja interrompido por isso.

In [8]:
class Valor:
    def __init__(self, data, progenitor=(), operador_mae="", rotulo=""):
        self.data = data
        self.progenitor = progenitor
        self.operador_mae = operador_mae
        self.rotulo = rotulo
        self.grad = 0

    def __str__(self):
        if self.rotulo:
            return f"Valor(data={self.data}), rotulo={self.rotulo})"
        else:
            return f"Valor(data={self.data})"

    def __repr__(self):
        data = self.data
        progenitor = self.progenitor
        operador_mae = self.operador_mae
        rotulo = self.rotulo

        argumentos = [f"{data=}"]

        if progenitor:
            argumentos.extend([f"{progenitor=}", f"{operador_mae=}"])
        if rotulo:
            argumentos.append(f"{rotulo=}")

        argumentos = ', '.join(argumentos)

        return f"Valor({argumentos})"

    def __add__(self, outro_valor):
        """Realiza a operação: self + outro_valor."""

        if not isinstance(outro_valor, Valor):
            outro_valor = Valor(outro_valor)

        progenitor = (self, outro_valor)
        data = self.data + outro_valor.data
        operador_mae = "+"
        resultado = Valor(data, progenitor, operador_mae)

        def propagar_adicao():
            self.grad += resultado.grad
            outro_valor.grad += resultado.grad

        resultado.propagar = propagar_adicao

        return resultado

    def __mul__(self, outro_valor):
        """Realiza a operação: self * outro_valor."""

        if not isinstance(outro_valor, Valor):
            outro_valor = Valor(outro_valor)

        progenitor = (self, outro_valor)
        data = self.data * outro_valor.data
        operador_mae = "*"
        resultado = Valor(data, progenitor, operador_mae)

        def propagar_multiplicacao():
            self.grad += resultado.grad * outro_valor.data # grad_filho * derivada filho em relação a mãe
            outro_valor.grad += resultado.grad * self.data

        resultado.propagar = propagar_multiplicacao

        return resultado

    def __pow__(self, expoente):
        """Realiza a operação: self ** expoente"""
        assert isinstance(expoente, (int, float))
        progenitor = (self, )
        data = self.data ** expoente
        operador_mae = f"**{expoente}"
        resultado = Valor(data, progenitor, operador_mae)

        def propagar_pow():
            self.grad += resultado.grad * (expoente * self.data ** (expoente - 1))

        resultado.propagar = propagar_pow

        return resultado

    def __truediv__(self, outro_valor):
        """Realiza a operação: self / outro_valor"""
        return self * outro_valor ** (-1)

    def __rtruediv__(self, outro_valor):
        """Realiza a operação: outro_valor / self"""
        return outro_valor * self ** (-1)

    def __neg__(self):
        """Realiza a operação: -self"""
        return self * -1

    def __sub__(self, outro_valor):
        """Realiza a operação: self - outro_valor"""
        return self + (-outro_valor)

    def __radd__(self, outro_valor):
        """Realiza a operação: outro_valor + self"""
        return self + outro_valor

    def __rsub__(self, outro_valor):
        """Realiza a operação: outro_valor - self"""
        return outro_valor + (-self)

    def __rmul__(self, outro_valor):
        """Realiza a operação: outro_valor * self"""
        return self * outro_valor

    def exp(self):
        """Realiza a operação: exp(self)"""
        progenitor = (self, )
        data = math.exp(self.data)
        operador_mae = "exp"
        resultado = Valor(data, progenitor, operador_mae)

        def propagar_exp():
            self.grad += resultado.grad * data

        resultado.propagar = propagar_exp

        return resultado

    def log(self):
        """Realiza a operação: log(self)"""
        progenitor = (self, )
        # Evita erro de domínio
        if self.data == 0:
            self_data = 10**(-8)
        else:
            self_data = self.data
        data = math.log(self_data)
        operador_mae = "log"
        resultado = Valor(data, progenitor, operador_mae)

        def propagar_log():
            self.grad += resultado.grad / self_data

        resultado.propagar = propagar_log

        return resultado

    def sig(self):
        """Realiza a operação: 1 / (1 + exp(-self))

        Esta operação é equivalente a "exp(self) / (exp(self) + 1)", porém tem
        maior estabilidade numérica.
        """

        return 1 / (1 + (-self).exp())

    def propagar(self):
        pass

    def propagar_tudo(self):

        self.grad = 1

        ordem_topologica = []

        visitados = set()

        def constroi_ordem_topologica(v):
            if v not in visitados:
                visitados.add(v)
                for progenitor in v.progenitor:
                    constroi_ordem_topologica(progenitor)
                ordem_topologica.append(v)

        constroi_ordem_topologica(self)

        for vertice in reversed(ordem_topologica):
            vertice.propagar()

### O Neurônio

Um _perceptron_:

1) recebe um ou mais dados de entrada;
2) modifica-os, ponderando-os com peso e adicionando a eles viés;
3) aplica sua função de ativação sobre eles, retornando um único dado.

Essa classe não foi alterada em relação ao material original; os parâmetros são inicializados aleatoriamente e modificados durante o treinamento, enquanto a função de ativação é a sigmoide.

Repare que o uso da sigmoide na última camada tem um motivo: lembra da discussão da troca de rótulo? Para que a previsão da rede possa ser avaliada durante o treinamento (ou para que esse dado numérico seja convertido em uma previsão de rótulo ao usar os dados de teste), é preciso que a rede forneça um dado que possa ser interpretado como uma probabilidade. Para isso, precisa estar no intervalo [0,1], que é justamente a imagem da função de ativação, pois a sigmoide é definida como 

$$f(x)=\frac{e^x}{e^x+1}$$

In [9]:
class Neuronio:
    def __init__(self, num_dados_entrada):
        self.vies = Valor(random.uniform(-1, 1))

        self.pesos = []
        for i in range(num_dados_entrada):
            self.pesos.append(Valor(random.uniform(-1, 1)))

    def __call__(self, x):

        assert len(x) == len(self.pesos)

        soma = 0
        for info_entrada, peso_interno in zip(x, self.pesos):
            soma += info_entrada * peso_interno

        soma += self.vies
        dado_de_saida = soma.sig()

        return dado_de_saida

    def parametros(self):
        return self.pesos + [self.vies]

### Camada de neurônios

Agora que definimos um neurônio, podemos colocar vários deles em paralelo:

1) neurônios da mesma camada recebem os mesmos dados: todos os da camada anterior, ou simplesmente os dados de entrada, se for a primeira camada;
2) realizam sua transformação **independentemente**, sem receber informação de qualquer neurônio que não os da camada anterior;
3) fornecem esse dado para cada neurônio da camada seguinte, ou o fornecem como dado de saída da rede, se for a última camada.
   
Essa parte do código também não foi alterada do material original.

In [10]:
class Camada:
    def __init__(self, num_neuronios, num_dados_entrada):
        neuronios = []

        for _ in range(num_neuronios):
            neuronio = Neuronio(num_dados_entrada)
            neuronios.append(neuronio)

        self.neuronios = neuronios

    def __call__(self, x):
        dados_de_saida = []

        for neuronio in self.neuronios:
            informacao = neuronio(x)
            dados_de_saida.append(informacao)

        if len(dados_de_saida) == 1:
            return dados_de_saida[0]
        else:
            return dados_de_saida

    def parametros(self):
        params = []

        for neuronio in self.neuronios:
            params_neuronio = neuronio.parametros()
            params.extend(params_neuronio)

        return params

### Rede MLP

Colocando-se várias camadas em série, constrói-se nossa rede neural:

1) Para cada atributo do dado, há um neurônio na camada de entrada;
2) O dado é transmitido para a primeira camada oculta, que o transforma e o transmite para a segunda, e assim sucessivamente;
3) O dado é transmitido da última camada oculta para a camada de saída, então cada neurônio desta fornece um dado de saída.

Esse código também não foi alterado.

In [11]:
class MLP:
    def __init__(self, num_dados_entrada, num_neuronios_por_camada):

        percurso = [num_dados_entrada] + num_neuronios_por_camada

        camadas = []

        for i in range(len(num_neuronios_por_camada)):
            camada = Camada(num_neuronios_por_camada[i], percurso[i])
            camadas.append(camada)

        self.camadas = camadas

    def __call__(self, x):
        for camada in self.camadas:
            x = camada(x)
        return x

    def parametros(self):
        params = []

        for camada in self.camadas:
            parametros_camada = camada.parametros()
            params.extend(parametros_camada)

        return params

Agora já podemos instanciar nossa MLP com a disposição de neurônios que quisermos!

In [12]:
CAMADAS_OCULTAS = [3, 2]
arquitetura_da_rede = CAMADAS_OCULTAS + [NUM_DADOS_DE_SAIDA]
minha_mlp = MLP(NUM_DADOS_DE_ENTRADA, arquitetura_da_rede)

## Treinamento

Já vimos que os parâmetros de cada neurônio são inicializados aleatoriamente, e que eles são usados para transformar os dados de entrada de forma arbitrária. É praticamente impossível que a tentativa inicial forneça um bom resultado, por isso é preciso modificar esses parâmetros até que a rede esteja razoavelmente adaptada aos dados e à relação entre eles.

O treinamento de uma MLP de classificação segue os mesmos passos de uma de regressão:

1) faz-se a previsão com os parâmetros atuais;
2) verifica-se o quanto ela é boa com uma função de perda;
3) calcula-se a influência de cada parâmetro nela usando _backpropagation_ (após os gradientes terem sido zerados);
4) tenta-se minimizá-la ao atualizar cada parâmetro no sentido contrário ao do gradiente.

A diferença fundamental está em qual função de perda será usada. Enquanto na classificação usávamos o MSE ou o RMSE para verificar o quão próxima a previsão estava dos dados reais, agora queremos analisar o quão alta é a probabilidade de ser escolhido o rótulo verdadeiro. Para isso, a função de perda usada geralmente é a _BINARY CROSS-ENTROPY_:

$$L = -(y_r\cdot log(y_p)+(1-y_r)\cdot log(1-y_p))$$

Em que $y_r$ é o rótulo real do alvo (codificado como 0 ou 1) e $y_p$ é a probabilidade prevista para o rótulo positivo. Assim, a função tende a zero quando a previsão é muito boa e a infinito quando é muito ruim.

In [13]:
def treinar(REDE, NUM_EPOCAS, TAXA_DE_APRENDIZADO, X_TREINO, Y_TREINO, v=False):

    for epoca in range(NUM_EPOCAS):
    
        # forward pass
        y_pred = []
        for exemplo in X_TREINO:
            previsao = REDE(exemplo)
            y_pred.append(previsao)
    
        # loss - BINARY CROSS-ENTROPY
        losses = []
        for yt, yp in zip(Y_TREINO, y_pred):
            logyp = yp.log()
            ytlogyp = yt*logyp
            I_yt = 1 - yt
            I_yp = 1 - yp
            logI_yp = I_yp.log()
            I_ytlogI_yp = I_yt*logI_yp
            menosL = ytlogyp + I_ytlogI_yp
            L = -menosL
            losses.append(L)
        loss = sum(losses)
    
        # zero grad
        for p in REDE.parametros():
            p.grad = 0
    
        # backpropagation
        loss.propagar_tudo()
    
        # atualiza parâmetros
        for p in REDE.parametros():
            p.data = p.data - p.grad * TAXA_DE_APRENDIZADO
    
        # mostra resultado (opcional)
        if v == True:
            print(epoca, loss.data)

    return REDE

Agora já podemos treinar nossa rede MLP classificadora!

In [14]:
argumentos_treino = {'REDE' : minha_mlp, 'NUM_EPOCAS' : 100, 'TAXA_DE_APRENDIZADO' : 2**(-8),
                  'X_TREINO' : X_treino_norm_, 'Y_TREINO' : y_treino_}
minha_mlp = treinar(**argumentos_treino)

### Previsão e Desempenho

A previsão de nossa rede neural não pode ser simplemente o retorno usando os dados de teste pois, no momento, esse valor é um número de 0 a 1, não necessariamente um deles! Para concluir o processo de classificação, é preciso estabelecer um limiar abaixo do qual a confiança no rótulo positivo atribui o negativo. Dessa forma, se usarmos o limiar 0.5:

* $y_p \geq 0.5 \implies$ Rede prevê rótulo 1 (True)
* $y_p < 0.5 \implies$ Rede prevê rótulo 0 (False)

O que equivale a simplesmente arredondar o dado. Mais interessante que isso: equivale a selecionar o rótulo mais provável (com um pequeno critério de desempate)! Esse pensamento será importante mais adiante, no caso multiclasse.

In [15]:
def prever(REDE, X_TESTE):

    y_pred = []
    
    for exemplo in X_TESTE:
        previsao = REDE(exemplo)
        y_pred.append(previsao.data)
    
    y_pred_bin = []
    for valor_previsto in y_pred:
        y_pred_bin.append(round(valor_previsto))

    return y_pred_bin

y_predito = prever(minha_mlp, X_teste_norm_)

Dessa forma, ao comparar os valores previstos com os reais, há quatro resultados possíveis:

* Verdadeiros positivos, quando o rótulo previsto é 1 e coincide com o real;
* Verdadeiros negativos, quando o rótulo previsto é 0 e coincide com o real;
* Falsos positivos, quando o rótulo previsto é 1 e difere do real;
* Falsos negativos, quando o rótulo previsto é 0 e difere do real.

Podemos, então, construir a matriz de confusão, que nada mais é que a contagem de todos esses casos, na forma de um dicionário: 

In [16]:
def comparar(Y_TESTE, Y_PRED):

    true0, true1, false0, false1 = 0, 0, 0, 0
    
    for yt, yp in zip(Y_TESTE, Y_PRED):
        if yt == 1:
            if yp == 1:
                true1 += 1
            elif yp == 0:
                false0 += 1
        elif yt == 0:
            if yp == 0:
                true0 += 1
            elif yp == 1:
                false1 += 1
    return {'Verdadeiros Positivos (VP)' : true1, 'Falsos Positivos (FP)' : false1,
            'Verdadeiros Negativos (VN)' : true0, 'Falsos Negativos (FN)' : false0}

confusão = comparar(y_teste_, y_predito)
conf = list(confusão.values())
confusão

{'Verdadeiros Positivos (VP)': 11,
 'Falsos Positivos (FP)': 0,
 'Verdadeiros Negativos (VN)': 60,
 'Falsos Negativos (FN)': 29}

Várias métricas diferentes podem ser usadas para avaliar esses dados, de acordo com seus objetivos[4][5]. Em um modelo ideal, que acertasse todas as classificações, as três métricas a seguir teriam valor 1:
### Acurácia
Equivale a qual fração das suas previsões coincidiu com o rótulo real.
$$Acc = \frac{VP+VN}{VP+FP+VN+FN}$$
A acurácia mede a sua frequência de acerto, sem se importar com a diferenciação dos rótulos. Por isso, é uma medida simples e válida, mas pouco útil com dados desbalanceados - ou seja, quando os rótulos não aparecem todos preticamente na mesma quantidade. Por exemplo, se o objetivo fosse dizer se uma pessoa porta ou não uma doença raríssima, chutar sempre "NÃO!" resultaria em uma acurácia altíssima, apesar de ser o modelo mais simples possível. Para dados balanceados e nos quais não há interesse especial em um rótulo específico, no entanto, a acurácia serve, e esse é essencialmente nosso caso. Vejamos a acurácia de nossa MLP:

In [17]:
def acc(vp, fp, vn, fn):
    return (vp+vn)/(vp+fp+vn+fn)

ACC = acc(*conf)
print('Acurácia:',ACC)

Acurácia: 0.71


### Acurácia Balanceada
Para lidar com dados *des*balanceados, podemos trazer o equilíbrio balanceando a métrica! A acurácia balanceada tira a média (relativa ao número de rótulos possíveis) das frequências com que os dados tinham um certo rótulo e você previu corretamente:
$$BalAcc = \frac{\frac{VP}{VP+FN}+\frac{VN}{VN+FP}}{2}$$
Se seus dados estiverem balanceados, $VP+FN~VN+FP$, então é esperado que a acurácia balanceada se aproxime da acurácia.

In [18]:
def balacc(vp, fp, vn, fn):
    frac1 = vp/(vp+fn)
    frac2 = vn/(vn+fp)
    return (frac1+frac2)/2

BALACC = balacc(*conf)
print('Acurácia Balanceada:',BALACC)

Acurácia Balanceada: 0.6375


### Precisão

Equivale a qual fração das suas previsões de rótulo positivo estavam corretas.

$$Pre = \frac{VP}{VP+FP}$$

A precisão mede a frequência com que você acerta ao prever determinado rótulo, não um rótulo qualquer, como a acurácia. Se isso for importante para você, ela pode ser uma boa métrica: por exemplo, se você tiver precisão muito baixa ao prever se um material tem ou não capacidade de supercondução, pode investir muito tempo e dinheiro investigando compostos que na verdade não são interessantes. No caso do nosso _dataset_, um falso positivo pode significar que um guerreiro incompetente encontrará sua morte ao ser enviado por engano para enfrentar o chefão!

In [19]:
def pre(vp, fp, vn, fn):
    try:
        return (vp)/(vp+fp)
    except ZeroDivisionError:
        print('Não foram previstos dados com esse rótulo')
        return 0

PRE = pre(*conf)
print('Precisão:',PRE)

Precisão: 1.0


### Taxa de Verdadeiro Positivo (_Recall_)
Equivale a qual fração dos seus dados realmente positivos foi prevista corretamente.
$$TVP = \frac{VP}{VP+FN}$$
A TVP mede a frequência com que os dados realmente tinham um determinado rótulo e você acertou; ao contrário da precisão, ela aumenta ao se minimizar os falsos negativos, não os falsos positivos. Em testes de infecções, por exemplo, é muito importante que essa taxa seja alta para não deixar passar indivíduos contagiosos - melhor deixar um indivíduo são de quarentena! Vejamos o _recall_ da rede neural:

In [20]:
def tvp(vp, fp, vn, fn):
    return (vp)/(vp+fn)

TVP = tvp(*conf)
print('Taxa de Verdadeiro Positivo:',TVP)

Taxa de Verdadeiro Positivo: 0.275


## Construção da rede neural com PyTorch
O código em Python puro funciona, mas é puramente didático; bibliotecas como o PyTorch permitem criar redes MLP de forma muito mais simples e com custo computacional melhor, possibilitando o uso de mais neurônios e mais épocas em menos tempo! Obviamente, um recurso tão poderoso teria suas exigências, e a primeira dela é que os dados estejam na forma de tensor[6]. Repare que não fiz a transformação do `y_teste`, mas apenas porque ele não é usado em nenhuma etapa de treinamento ou previsão da rede, apenas para calcular o desempenho depois.

In [21]:
X_treino_norm_tensor = torch.tensor(X_treino_norm, dtype=torch.float32)
y_treino_tensor = torch.tensor(y_treino_, dtype=torch.float32).view(-1, 1)

X_teste_norm_tensor = torch.tensor(X_teste_norm, dtype=torch.float32)

Ainda mudando o mínimo possível do material do Prof. Daniel R. Cassar, criaremos uma classe correspondente à rede MLP. As funções de ativação em cada camada ainda são todas sigmoides; optei por deixar a última função modificável para poder reaproveitar o código mais adiante.

In [22]:
class torch_MLP(nn.Module):
    def __init__(self, num_dados_entrada, neuronios_c1, neuronios_c2, num_targets, final=nn.Sigmoid()):
        super().__init__()

        self.camadas = nn.Sequential(
            nn.Linear(num_dados_entrada, neuronios_c1),
            nn.Sigmoid(),
            nn.Linear(neuronios_c1, neuronios_c2),
            nn.Sigmoid(),
            nn.Linear(neuronios_c2, num_targets),
            final
        )

    def forward(self, x):
        x = self.camadas(x)
        return x

Já podemos instanciar a rede com PyTorch! O exemplo a seguir tem duas camadas ocultas, com 5 e 4 neurônios, nessa ordem.

In [23]:
torch_mlp = torch_MLP(NUM_DADOS_DE_ENTRADA, 5, 4, NUM_DADOS_DE_SAIDA)

Agora precisamos definir a etapa de treino, com os mesmos parâmetros de antes. Mais uma vez, a diferença do material original está na escolha de função de perda; essa etapa foi construída de modo que, ao chamá-la, essa função pode ser escolhida, então esse código pode ser reaproveitado.

In [24]:
def treinar_torch(REDE, NUM_EPOCAS, TAXA_DE_APRENDIZADO, X_TREINO, Y_TREINO, LOSS, v=False):

    otimizador = optim.SGD(REDE.parameters(), lr=TAXA_DE_APRENDIZADO)
    
    REDE.train()
    
    for epoca in range(NUM_EPOCAS):
    
        # forward pass
        y_pred = REDE(X_TREINO)
    
        # zero grad
        otimizador.zero_grad()
    
        # loss
        loss = LOSS(y_pred, Y_TREINO)
    
        # backpropagation
        loss.backward()
    
        # atualiza parâmetros
        otimizador.step()
    
        # mostra resultado (opcional)
        if v == True:
            print(epoca, loss.data)
            
    REDE.eval()

    return REDE

Podemos treinar nossa rede; perceba como é mais rápido que o código em Python puro! A função de perda, `nn.BCELoss()`, é justamente a **B**inary **C**ross-**E**ntropy.

In [25]:
argumentos_torch = {'REDE' : torch_mlp, 'NUM_EPOCAS' : 1250, 'TAXA_DE_APRENDIZADO' : 0.5,
                    'X_TREINO' : X_treino_norm_tensor, 'Y_TREINO' : y_treino_tensor, 'LOSS' : nn.BCELoss(), 'v' : False}

torch_mlp = treinar_torch(**argumentos_torch)

A função de previsão é estabelecida de modo que receba os dados de teste na forma de tensor, usada pela rede, mas retorne uma lista de zeros e uns, como antes.

In [26]:
def prever_torch(REDE, X_TESTE):
    with torch.no_grad():
        y_pred =REDE(X_TESTE).tolist()
    y_pred_bin = []
    for lista in y_pred:
        y_pred_bin.append(round(lista[0]))
    return y_pred_bin

y_predito_torch = prever_torch(torch_mlp, X_teste_norm_tensor)

Assim, podemos usar as mesmas funções de comparação e métricas de desempenho de anteriormente:

_Obs:_ como é um _dataset_ simples e não trabalhamos com dados de validação ou otimização de hiperparâmetros, pode haver resultados esquisitos.

In [27]:
confusão_torch = comparar(y_teste_, y_predito_torch)
conf_torch = list(confusão_torch.values())
confusão_torch

{'Verdadeiros Positivos (VP)': 40,
 'Falsos Positivos (FP)': 3,
 'Verdadeiros Negativos (VN)': 57,
 'Falsos Negativos (FN)': 0}

In [28]:
print('Acurácia:', acc(*conf_torch))
print('Acurácia Balanceada:', balacc(*conf_torch))
print('Precisão:', pre(*conf_torch))
print('Taxa de Verdadeiro Positivo:', tvp(*conf_torch))

Acurácia: 0.97
Acurácia Balanceada: 0.975
Precisão: 0.9302325581395349
Taxa de Verdadeiro Positivo: 1.0


# Classificação Multiclasse

Quando o alvo tem três ou mais rótulos possíveis (por exemplo: a estrutura cristalina pode ser ortorrômbica, cúbica, cúbica de corpo centrado, cúbica de face centrada etc.), é um problema de classificação multiclasse. Com nosso _dataset_, podemos tentar prever a classe ('Class') de um personagem de acordo com seus atributos. Para isso, será usado o `PyTorch`, então precisamos transformar nossos dados em tensores numéricos. Assim, cada rótulo do alvo será codificado como um número inteiro do tipo `long`, exigido pelas funções que usaremos.

In [29]:
display(df.reindex(ATRIBUTOS+['Class'], axis=1))

y_mul_treino = df_treino.reindex(['Class'], axis=1)
y_mul_teste = df_teste.reindex(['Class'], axis=1)
y_mul_treino_ = y_mul_treino.to_numpy().ravel().tolist()
y_mul_teste_ = y_mul_teste.to_numpy().ravel().tolist()

classes = list(set(y_mul_treino_))
num_classes = len(classes)
dic_classes = dict(zip(classes, range(num_classes)))

y_mul_treino_codificado,  y_mul_teste_codificado = [], []
for item in y_mul_treino_:
    y_mul_treino_codificado.append(dic_classes[item])
for item in y_mul_teste_:
    y_mul_teste_codificado.append(dic_classes[item])
    
y_mul_treino_tensor = torch.tensor(y_mul_treino_codificado, dtype=torch.long).squeeze()
# .squeeze() remove todas as dimensões 1 do tensor; necessário para função de perda (deve coincidir com o shape dos dados de X treino)

print('Código:', dic_classes)

,Armor,Weapon,Physical,Magic,Level,Class
0,45,83,45,89,77,Magician
1,48,88,82,65,71,Warrior
2,65,22,71,4,37,Tank
3,68,3,7,91,39,Sorcerer
4,68,9,49,21,27,Tank
...,...,...,...,...,...,...
995,27,43,73,90,67,Battlemage
996,49,69,81,30,50,Knight
997,72,63,19,72,61,Magician
998,55,37,75,27,66,Battlemage


Código: {'Sorcerer': 0, 'Tank': 1, 'Magician': 2, 'Warrior': 3, 'Battlemage': 4, 'Knight': 5}


Agora que os dados estão na forma correta, só falta fazer a rede MLP de classificação multiclasse!

### Só falta tudo??

Exatamente! Falta tudo, mas o que falta é pouco, porque já definimos as classes e funções necessárias para instanciar e treinar uma MLP - e, mais importante, já entendemos as ideias-base para as redes de classificação! Com pequenas mudanças, podemos resolver o caso multiclasse.

## As ideias-base

* Em uma MLP classificadora, a última camada devolve valores numéricos relacionados às probabilidades dos rótulos;
* Esses valores são usados na função de perda, minimizada pela atualização dos pesos e vieses durante o treinamento;
* Na previsão, o dado de saída da rede é o rótulo com maior probabilidade;
* Com a comparação entre rótulos reais e previstos, constrói-se a matriz de confusão e as métricas de desempenho.

## No caso binário

* Apenas um valor era retornado - a probabilidade do rótulo positivo -, portanto havia um neurônio na camada de saída;
* A função sigmoide era usada na última camada para transformar o dado bruto em uma probabilidade;
* A função de perda era a entropia cruzada _binária_;
* A matriz de confusão e as métricas de desempenho consideravam o caso _binário_.

## No caso multiclasse 

* Será retornado um valor para cada rótulo possível - a probabilidade de cada um deles -, portanto haverá um neurônio na camada de saída para cada rótulo possível;
* Outra função - Softmax - será usada para transformar os dados brutos em uma distribuição discreta de probabilidade [7]
  * mais precisamente, o logaritmo dessa função, puramente para simplificar a implementação da função de perda com `PyTorch`;
* A função de perda será outra - Log-verossimilhança negativa -, adequada para várias classes;
* A matriz de confusão e as métricas de desempenho considerarão todas as combinações possíveis de rótulo real e previsto.

# As novas funções
## Softmax
A função softmax transforma um vetor (ou seja, uma lista qualquer de números) em uma distribuição de probabilidades (ou seja, uma lista de números, cada um deles no intervalo [0,1], de modo que a soma de todos eles seja 1) da seguinte forma: cada componente do vetor é substituída por sua exponencial, dividida pela soma das exponenciais de todas as componentes do vetor. Veja o exemplo:

In [30]:
def softmax(vetor):
    # Novo vetor com as exponenciais de cada componente:
    exp_vetor = []
    for componente in vetor:
        exp_vetor.append(math.e**componente)
    # Soma das exponenciais de cada componente:
    soma_exp = sum(exp_vetor)
    # Novo vetor normalizado pela soma das exponenciais:
    softmax_vetor = []
    for exp_componente in exp_vetor:
        softmax_vetor.append(exp_componente/soma_exp)
    return softmax_vetor

vetor_exemplo = [0,1,0]
softmax_exemplo = softmax(vetor_exemplo)
print(f'Softmax({vetor_exemplo}) = {softmax_exemplo}')

Softmax([0, 1, 0]) = [0.21194155761708547, 0.5761168847658291, 0.21194155761708547]


## LogSoftmax

Transforma um vetor de modo que cada componente é substituída pelo logaritmo natural da componente da softmax aplicada àquele vetor. Dessa forma, se quisermos recuperar uma distribuição de probabilidade, basta construir um vetor em que cada componente seja $e$ elevado à componente encontrada com Logsoftmax.

Então por que não usar Softmax logo de cara? Apenas porque, por motivo de custo e estabilidade numérica, o `PyTorch` tem funções de perda que usam `nn.LogSoftmax()`: a `nn.CrossEntropyLoss()`, usada para classificação multiclasse, equivale a receber os dados brutos (_logits_) e aplicar `nn.LogSoftmax()` e `nn.NLLLoss()` logo em seguida. Se optássemos por isso, a última camada não deveria ter função de ativação, e para obter as probabilidades na etapa de previsão dever-se-ia usar `nn.Softmax()`, mas preferi a abordagem com log para manter o paralelismo com a rede binária.

In [31]:
def logsoftmax(vetor):
    # Novo vetor com as softmax de cada componente:
    softmax_vetor = softmax(vetor)
    # Novo vetor com o log natural de cada componente do vetor softmax:
    logsoftmax_vetor = []
    for componente in softmax_vetor:
        logsoftmax_vetor.append(math.log(componente))
    return logsoftmax_vetor

logsoftmax_exemplo = logsoftmax(vetor_exemplo)
print(f'LogSoftmax({vetor_exemplo}) = {logsoftmax_exemplo}')

LogSoftmax([0, 1, 0]) = [-1.551444713932051, -0.5514447139320511, -1.551444713932051]


## Log-Verossimilhança Negativa (`nn.NLLLoss()`)
A função de perda na verdade é bem simples: o logaritmo da softmax fornece uma "lista" com o logaritmo natural da probabilidade de cada rótulo. Sabendo qual é o rótulo verdadeiro, a log-verossimilhança negativa nada mais é que o valor na lista relacionado a esse rótulo, mas com o sinal trocado, pois queremos minimizar a perda ao aumentar a probabilidade (e portanto seu log) de se escolher o rótulo correto.

In [32]:
def nllloss(vetor_logsoftmax, valor_correto):
    return -(vetor_logsoftmax[valor_correto])

nllloss_exemplo = nllloss(logsoftmax_exemplo, valor_correto=1)
print(f'NLLLoss({logsoftmax_exemplo}, valor_correto=1) = {nllloss_exemplo}')

NLLLoss([-1.551444713932051, -0.5514447139320511, -1.551444713932051], valor_correto=1) = 0.5514447139320511


Para vários dados sendo avaliados juntos, por padrão, `nn.NLLoss()` fornece a média das log-verossimilhanças negativas. O `torch`permite que você mude isso para uma soma ou atribua pesos às classes; esse último recurso é útil para dados desbalanceados, mas não será necessário para nós.

In [33]:
def NLLLoss(lista_logsoftmax, lista_valores, somar=False):
    tamanho = len(lista_valores)
    assert len(lista_logsoftmax) == tamanho, 'Os dados devem ter o mesmo tamanho'
    soma = 0
    for vetor_logsoftmax, valor_correto in zip(lista_logsoftmax, lista_valores):
        soma += nllloss(vetor_logsoftmax, valor_correto)
    if somar == True:
        return soma
    return soma / tamanho
    
lista_exemplo = [logsoftmax_exemplo]*2
reais_exemplo = [1,2]
NLL = NLLLoss(lista_exemplo, reais_exemplo)
print('Lista com logsoftmax:', lista_exemplo)
print('Rótulos Reais:', reais_exemplo)
print(f'NLLLoss = {NLL}')

Lista com logsoftmax: [[-1.551444713932051, -0.5514447139320511, -1.551444713932051], [-1.551444713932051, -0.5514447139320511, -1.551444713932051]]
Rótulos Reais: [1, 2]
NLLLoss = 1.051444713932051


## Construindo e Treinando a Rede Classificadora Multiclasse
_Obs:_ o argumento 1 é colocado na `nn.LogSoftmax` para indicar a dimensão em que os dados devem ser uma distribuição de probabilidade - queremos que a soma das probabilidades de cada _atributo_ seja 1 para todos os dados, não o oposto!

In [34]:
multi_mlp = torch_MLP(NUM_DADOS_DE_ENTRADA, 6, 6, num_classes, final=nn.LogSoftmax(1))

argumentos_multi = {'REDE' : multi_mlp, 'NUM_EPOCAS' : 1000, 'TAXA_DE_APRENDIZADO' : 0.5,
                    'X_TREINO' : X_treino_norm_tensor, 'Y_TREINO' : y_mul_treino_tensor, 'LOSS' : nn.NLLLoss()}

multi_mlp = treinar_torch(**argumentos_multi)

### Etapa de Previsão

Por enquanto, nossa rede está retornando seis valores numéricos, cada qual equivalente ao **logaritmo natural** da probabilidade de um rótulo. Assim como no caso binário, temos de fazer a função escolher o rótulo com a maior **probabilidade**. De fato, um log maior quer dizer que a probabilidade é maior, então grande parte da função abaixo é dispensável - bastaria retornar a lista com os índices do máximo de cada lista em `y_pred`. No entanto, como esse _notebook_ preza mais pela didática do que pela eficiência, optei por construir as distribuições de probabilidade. Assim, ao usar o argumento `v=True`, o usuário pode verificar que, de fato, cada resposta oferecida pela rede foi matematicamente válida.

In [35]:
def prever_multi(REDE, X_TESTE, v = False):
    # Previsão com a rede neural
    with torch.no_grad():
        y_pred = REDE(X_TESTE).tolist()
    # Transformação em distribuições de probabilidade
        e_y_pred = []
        for lis in y_pred:
            nova_lista = []
            for logaritmo in lis:
                probabilidade = math.e**logaritmo
                nova_lista.append(probabilidade)
            e_y_pred.append(nova_lista)
    # Escolha do rótulo com maior probabilidade
        y_pred_class = []
        for lis in e_y_pred:
            y_pred_class.append(lis.index(max(lis)))
    # Verificação didática
        if v == True:
            valores_previstos = []
            somas_probabilidades = []
            for lista_probabilidades in e_y_pred:
                valores_previstos.extend(lista_probabilidades)
                somas_probabilidades.append(sum(lista_probabilidades))
            # Cada valor previsto estava contido em [0,1]?
            print('Intervalo das probabilidades:', (round(min(valores_previstos),3),round(max(valores_previstos),3)))
            # Cada lista era uma distribuição de probabilidades (somava 1)?
            print('Intervalo das somas das probabilidades:', (round(min(somas_probabilidades),3),round(max(somas_probabilidades),3)))
            # Várias categorias estavam sendo previstas?
            print('Categorias efetivamente previstas:', set(y_pred_class))
    return y_pred_class

In [36]:
y_predito_multi = prever_multi(multi_mlp, X_teste_norm_tensor, v=True)

Intervalo das probabilidades: (0.002, 0.758)
Intervalo das somas das probabilidades: (1.0, 1.0)
Categorias efetivamente previstas: {0, 1, 2, 3, 4, 5}


# Desempenho de uma Rede Multiclasse

Existem análogos das métricas que já vimos para o caso binário. Inclusive, todas as funções que serão definidas nessa etapa podem ser usadas no caso binário, mas aqui estão no caso geral!

## Matriz de Confusão

Ela computará a quantidade de cada uma das combinações possíveis de rótulo real e previsto. Com nossos dados, são 36 entradas no dicionário! Felizmente, bibliotecas prontas fornecem matrizes mais apresentáveis, mas para explicar as métricas de desempenho em Python puro, continuarei nesse caminho.

In [37]:
def matriz_confusão(real, prev):
    # Número de dados
    tamanho = len(real)
    assert len(prev) == tamanho, "Os dados devem ter a mesma dimensão"
    # Rótulos possíveis
    valores_reais = sorted(set(real+prev))
    # Criação do dicionário
    matriz = {}
    # Chaves = combinações possíveis; Valores = contagem (inicialmente 0)
    for valor_real in valores_reais:
        for valor_previsto in valores_reais:
            matriz[(valor_real, valor_previsto)] = 0
    # Contagem das combinações
    combinações = list(zip(real,prev))
    for combinação in combinações:
        matriz[combinação] += 1
    return matriz

In [38]:
matriz_multi = matriz_confusão(y_mul_teste_codificado, y_predito_multi)

## Acurácia
Ainda é a frequência total de acertos - com a diferença de que existem mais formas de acertar e errar!

In [39]:
def Acurácia(matriz):
    # Conta quantas previsões foram feitas
    total = sum(list(matriz.values()))
    # Conta em quantas delas o valor real foi igual ao previsto
    corretos = 0
    for tupla in matriz:
        if tupla[0] == tupla[1]:
            corretos += matriz[tupla]
    # Retorna a frequência de acertos
    return corretos/total

In [40]:
Acurácia(matriz_multi)

0.54

## Precisão e Recall
De forma análoga ao caso binário, podem-se calcular essas métricas tomando como base qualquer um dos rótulos verdadeiros! Fundamentalmente, até a "fórmula" é a mesma, bastando substituir:
* Verdadeiros Positivos *por* Qualquer um que tinha de fato aquele rótulo e foi previsto corretamente;
* Falsos Positivos *por* Qualquer um que NÃO tinha de fato aquele rótulo, mas foi previsto como se tivesse;
* Falsos Negativos *por* Qualquer um que tinha de fato aquele rótulo e NÃO foi previsto corretamente.

In [41]:
def Precisão(matriz, rótulo):
    # "verdadeiros positivos"
    verdadeiro = matriz[(rótulo, rótulo)]
    # "verdadeiros positivos" + "falsos positivos"
    previsto = 0
    for tupla in matriz:
        if tupla[1] == rótulo:
            previsto += matriz[tupla]
    if previsto == 0:
        print('Não foram previstos dados com esse rótulo')
        return 0
    # "verdadeiros positivos" / ("verdadeiros positivos" + "falsos positivos")
    return verdadeiro / previsto

In [42]:
for i in range(6):
    print(f'Precisão (rótulo {i}) = {Precisão(matriz_multi, i)}')

Precisão (rótulo 0) = 0.5217391304347826
Precisão (rótulo 1) = 0.7083333333333334
Precisão (rótulo 2) = 0.45454545454545453
Precisão (rótulo 3) = 0.6
Precisão (rótulo 4) = 1.0
Precisão (rótulo 5) = 0.2


In [43]:
def Recall(matriz, rótulo):
    # "verdadeiros positivos"
    verdadeiro = matriz[(rótulo, rótulo)]
    # "verdadeiros positivos" + "falsos negativos"
    real_rotulado = 0
    for tupla in matriz:
        if tupla[0] == rótulo:
            real_rotulado += matriz[tupla]
    # "verdadeiros positivos" / ("verdadeiros positivos" + "falsos negativos")
    return verdadeiro / real_rotulado

In [44]:
for i in range(6):
    print(f'Recall (rótulo {i}) = {Recall(matriz_multi, i)}')

Recall (rótulo 0) = 0.75
Recall (rótulo 1) = 0.7391304347826086
Recall (rótulo 2) = 0.7692307692307693
Recall (rótulo 3) = 0.6666666666666666
Recall (rótulo 4) = 0.0625
Recall (rótulo 5) = 0.14285714285714285


## Acurácia Balanceada
As mesmas substituições mentais que fizemos são usadas na acurácia balanceada, mas ela considera todos os rótulos possíveis - imagine que você vai considerar cada um deles como o caso positivo uma vez. Ela considera a fração entre todos os casos positivos para esse rótulo e que você previu corretamente e todos os casos que realmente eram positivos, independentemente de sua previsão. Então, ela tira a média dessas frações em relação ao número de rótulos possíveis. Ou seja: **a acurácia balanceada equivale à média de todos os _recall_**!

In [45]:
def AcuráciaBalanceada(matriz):
    # Rótulos
    rótulos = []
    for tupla in matriz:
        rótulos.append(tupla[0])
    rótulos = set(rótulos)
    # Soma dos recall
    numerador = 0
    for Rótulo in rótulos:
        numerador += Recall(matriz, Rótulo)
    # Média pelo número de rótulos
    return numerador / len(rótulos)

In [46]:
AcuráciaBalanceada(matriz_multi)

0.5217308355895313

# Função de Perda Ponderada

Como já dito, o `torch`permite que você mude a influência de cada classe na `nn.NLLLoss` ao adicionar como argumento um tensor com um peso para cada classe. É muito comum adotar essa estratégia para dados desbalanceados, o que não é o nosso caso, mas isso não quer dizer que não podemos fazer isso!
Façamos isso de duas formas: com pesos proporcionais à frequência do rótulo e com pesos inversamente proporcionais à frequência do rótulo, sem mudar outros parâmetros de nossa rede sem pesos.

In [47]:
lista_pesos = []
lista_inversos = []
for i in range(num_classes):
    contagem = y_mul_treino_codificado.count(i)
    lista_pesos.append(contagem)
    lista_inversos.append(1/contagem)
pesos = torch.tensor(lista_pesos, dtype=torch.float32)
inversos = torch.tensor(lista_inversos, dtype=torch.float32)

## Pesos proporcionais à frequência

In [48]:
mlp_ponderada = torch_MLP(NUM_DADOS_DE_ENTRADA, 6, 6, num_classes, final=nn.LogSoftmax(1))

argumentos_ponderada = {'REDE' : mlp_ponderada, 'NUM_EPOCAS' : 1000, 'TAXA_DE_APRENDIZADO' : 0.5,
                    'X_TREINO' : X_treino_norm_tensor, 'Y_TREINO' : y_mul_treino_tensor, 'LOSS' : nn.NLLLoss(weight=pesos)}

mlp_ponderada = treinar_torch(**argumentos_ponderada)

y_predito_ponderada = prever_multi(mlp_ponderada, X_teste_norm_tensor, v=True)
matriz_ponderada = matriz_confusão(y_mul_teste_codificado, y_predito_ponderada)
print('Acurácia =',Acurácia(matriz_ponderada))
print('Acurácia Balanceada =',AcuráciaBalanceada(matriz_ponderada))

Intervalo das probabilidades: (0.0, 0.922)
Intervalo das somas das probabilidades: (1.0, 1.0)
Categorias efetivamente previstas: {0, 1, 2, 3, 4, 5}
Acurácia = 0.55
Acurácia Balanceada = 0.5704519916476438


## Pesos inversamente proporcionais à frequência

In [49]:
mlp_inversa = torch_MLP(NUM_DADOS_DE_ENTRADA, 6, 6, num_classes, final=nn.LogSoftmax(1))

argumentos_inversa = {'REDE' : mlp_inversa, 'NUM_EPOCAS' : 1000, 'TAXA_DE_APRENDIZADO' : 0.5,
                    'X_TREINO' : X_treino_norm_tensor, 'Y_TREINO' : y_mul_treino_tensor, 'LOSS' : nn.NLLLoss(weight=inversos)}

mlp_inversa = treinar_torch(**argumentos_inversa)

y_predito_inversa = prever_multi(mlp_inversa, X_teste_norm_tensor, v=True)
matriz_inversa = matriz_confusão(y_mul_teste_codificado, y_predito_inversa)
print('Acurácia =',Acurácia(matriz_inversa))
print('Acurácia Balanceada =',AcuráciaBalanceada(matriz_inversa))

Intervalo das probabilidades: (0.002, 0.56)
Intervalo das somas das probabilidades: (1.0, 1.0)
Categorias efetivamente previstas: {0, 1, 2, 3, 4, 5}
Acurácia = 0.42
Acurácia Balanceada = 0.40874343048256095


Como nossos dados eram bem balanceados, não fez tanta diferença, mas vamos raciocinar:

Faz bastante sentido relacionar cada peso à frequência com que o rótulo aparece nos dados de treino. Pesos maiores aumentarão a influência na função de perda; tendo isso em vista, será melhor atribuir pesos maiores aos rótulos mais frequentes ou aos menos frequentes? 

Se você atribuir um peso pequeno a um rótulo pouco frequente, a influência dele, que já não era grande, será ainda menor. Dessa forma, a perda não melhorará praticamente nada se esse rótulo for ignorado! Além disso, atribuir um peso grande a uma classe frequente tornará ainda mais desfavorável deixar de prevê-la, de modo que se favorece um modelo que a preveja com ainda mais frequência!

Se você atribuir um peso maior a esse rótulo, no entanto, sua função de perda será grande quando você deixar de prevê-lo. Dessa forma, você pode "forçar" sua rede a aprender a incluir essa posibilidade de rótulo! Além disso, um peso menor em uma classe mais frequente tornará deixar de prevê-la menos prejudicial para a perda, favorecendo a previsão de outros rótulos.

Por isso, "A maneira mais comum de implementar uma função de perda ponderada é atribuir maior peso à classe minoritária e menor peso à classe majoritária. Os pesos podem ser inversamente proporcionais à frequência das classes, de modo que a classe minoritária receba maior peso e a classe majoritária receba menor peso.[8]"

# Conclusão
MLPs de regressão, classificação binária e classificação multiclasse diferem no formato de saída dos dados (para coincidir com o do alvo), sendo que as redes de classificação se valem bastante da ideia de atribuir uma probabilidade a cada rótulo; a maior distinção ocorre na escolha das funções da última camada, portanto, além das funções de perda e métricas de desempenho. Essas últimas podem ser adaptadas para lidar com dados desbalanceados (por exemplo, ponderando-se a função de perda e balanceando-se a acurácia), além de precisarem se adequar ao interesse da previsão - não vá sair usando acurácia para prever doenças raras!

No mais, a estrutura e a lógica por trás das MLPs se preserva. Note que, como já observado na introdução, este _notebook_ não contém todos os assuntos interessantes do tema - existem, inclusive, muitas outras formas de lidar com dados desbalanceados e métricas de desempenho que sequer foram mencionados aqui -, mas espera-se que ele ajude no entendimento inicial sobre problemas de classificação e como usar redes neurais neles.

## Referências

[1] Dataset didático: https://www.kaggle.com/datasets/jjasser/rpg-character-attributes-dataset

[2] Notebook Didático - Multilayer Perceptron em Python Puro (Prof. Daniel Cassar): https://ilumcnpem-my.sharepoint.com/:u:/r/personal/daniel_cassar_ilum_cnpem_br/Documents/Material%20Did%C3%A1tico/Redes%20Neurais/ATP-303%20NN%202.4%20-%20Multilayer%20Perceptron%20em%20Python%20puro.ipynb?csf=1&web=1&e=ZAiPxq

[3] Notebook Didático - Construindo e Treinando Redes Neurais com PyTorch e Lightning (Prof. Daniel Cassar): https://ilumcnpem-my.sharepoint.com/:u:/r/personal/daniel_cassar_ilum_cnpem_br/Documents/Material%20Did%C3%A1tico/Redes%20Neurais/ATP-303%20NN%203.2%20-%20Construindo%20e%20treinando%20redes%20neurais%20com%20PyTorch%20e%20Lightning.ipynb?csf=1&web=1&e=L2OBc4

[4] Google Developers - Classificação: acurácia, recall, precisão e métricas relacionadas. Disponível em: https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall?hl=pt-br

[5] Margherita Grandini, Enrico Bagli, Giorgio Visani - METRICS FOR MULTI-CLASS CLASSIFICATION: AN OVERVIEW. Disponível em: https://arxiv.org/pdf/2008.05756

[6] Documentação das redes neurais do PyTorch: https://docs.pytorch.org/docs/stable/nn.html

[7] Patrick Loeber - PyTorch Tutorial 11 - Softmax and Cross Entropy. Disponível em: https://www.youtube.com/watch?v=7q7E91pHoW4

[8] Hengtao Tantai - Use weighted loss function to solve imbalanced data classification problems. Disponível em: https://medium.com/@zergtant/use-weighted-loss-function-to-solve-imbalanced-data-classification-problems-749237f38b75

[9] Diálogo com o ChatGPT: https://chatgpt.com/share/6a0281bd-e9e8-83e9-9003-3606638cb48b